# 🎤 07 · 大数据面试八股与高频题

> 最后冲刺：把**体系/存储/计算/实时/数仓/管道/平台**的考点串成速答体系，
> 并手写两个高频手撕（TopN、两阶段聚合处理倾斜）。

## 1️⃣ 30 秒框架串讲

In [1]:
print('''一条数据从产生到应用的全链路：
  业务库/日志 -> 采集(Kafka/Flume) -> 存储(HDFS/湖) -> 离线计算(Hive/Spark)
               -> 数仓分层(ODS->DWD->DWS->ADS) -> 报表/算法特征
   另加实时链路：-> Flink 实时计算 -> 实时指标/风控
  调度(Airflow/DolphinScheduler) + 血缘 + 质量 + 权限 贯穿始终
''')

一条数据从产生到应用的全链路：
  业务库/日志 -> 采集(Kafka/Flume) -> 存储(HDFS/湖) -> 离线计算(Hive/Spark)
               -> 数仓分层(ODS->DWD->DWS->ADS) -> 报表/算法特征
   另加实时链路：-> Flink 实时计算 -> 实时指标/风控
  调度(Airflow/DolphinScheduler) + 血缘 + 质量 + 权限 贯穿始终



## 2️⃣ 高频题速答（一）体系与存储

In [2]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

In [3]:
print('''1) Hadoop 三件套？ -> HDFS(存储)+YARN(资源)+MapReduce(计算)；
2) HDFS 副本策略？  -> 3 副本：同机架 2/另一机架 1（默认策略描述）；
3) 小文件问题？     -> 大量小文件=海量元数据；合并/归档/湖格式；
4) 存算分离？       -> 存储与计算独立扩缩容；云原生趋势；
5) 冷热数据？       -> 冷数据低成本存储（对象存储），热数据快；
6) 数据湖 vs 数仓？ -> 湖：原始格式+ACID（Iceberg/Hudi）；仓：治理后分层。
''')

1) Hadoop 三件套？ -> HDFS(存储)+YARN(资源)+MapReduce(计算)；
2) HDFS 副本策略？  -> 3 副本：同机架 2/另一机架 1（默认策略描述）；
3) 小文件问题？     -> 大量小文件=海量元数据；合并/归档/湖格式；
4) 存算分离？       -> 存储与计算独立扩缩容；云原生趋势；
5) 冷热数据？       -> 冷数据低成本存储（对象存储），热数据快；
6) 数据湖 vs 数仓？ -> 湖：原始格式+ACID（Iceberg/Hudi）；仓：治理后分层。



## 3️⃣ 高频题速答（二）计算与实时

In [4]:
print('''7) Spark vs MapReduce？   -> 内存计算+DAG，快 10-100x；
8) RDD 血缘有什么用？     -> 容错重算/故障恢复；
9) 宽窄依赖？            -> 窄=流水线无 shuffle；宽=shuffle；
10) Flink watermark？     -> 处理乱序；低水位=等迟到数据；
11) 窗口类型？           -> 滚动/滑动/会话；
12) 精确一次语义？       -> 幂等+事务/两阶段提交。
''')

7) Spark vs MapReduce？   -> 内存计算+DAG，快 10-100x；
8) RDD 血缘有什么用？     -> 容错重算/故障恢复；
9) 宽窄依赖？            -> 窄=流水线无 shuffle；宽=shuffle；
10) Flink watermark？     -> 处理乱序；低水位=等迟到数据；
11) 窗口类型？           -> 滚动/滑动/会话；
12) 精确一次语义？       -> 幂等+事务/两阶段提交。



## 4️⃣ 高频题速答（三）数仓与管道

In [5]:
print('''13) 数仓四层？         -> ODS/DWD/DWS/ADS；
14) 星型 vs 雪花？      -> 冗余/查询快 vs 规范/join 多；
15) 数据倾斜怎么办？    -> 两阶段聚合/打散 key/随机前缀；
16) Kafka 为什么快？    -> 顺序写+零拷贝+批量压缩；
17) 消息积压怎么解？    -> 扩分区扩消费者/批量/降级；
18) 慢变维 SCD？        -> Type1 覆盖/Type2 版本化。
''')

13) 数仓四层？         -> ODS/DWD/DWS/ADS；
14) 星型 vs 雪花？      -> 冗余/查询快 vs 规范/join 多；
15) 数据倾斜怎么办？    -> 两阶段聚合/打散 key/随机前缀；
16) Kafka 为什么快？    -> 顺序写+零拷贝+批量压缩；
17) 消息积压怎么解？    -> 扩分区扩消费者/批量/降级；
18) 慢变维 SCD？        -> Type1 覆盖/Type2 版本化。



## 5️⃣ 手撕一：全局 TopN（分治思路）

In [6]:
# 每分区先取局部 TopN，再归并取全局 TopN
def topn_partition(nums, n):
    return sorted(nums, reverse=True)[:n]

partitions = [[5, 9, 3], [10, 2, 7], [8, 1, 6]]
k = 3
local = [topn_partition(p, k) for p in partitions]
print('各分区局部 Top3:', local)
global_top = topn_partition([x for p in local for x in p], k)
print('全局 Top3:', global_top)
assert global_top == [10, 9, 8]
print('✅ TopN 分治法正确（局部降量 -> 归并）')

各分区局部 Top3: [[9, 5, 3], [10, 7, 2], [8, 6, 1]]
全局 Top3: [10, 9, 8]
✅ TopN 分治法正确（局部降量 -> 归并）


## 6️⃣ 手撕二：数据倾斜两阶段聚合

In [7]:
# 普通 groupBy 会 skew：某 key 集中在 1 个 task
# 两阶段：先加随机前缀局部聚合，再去前缀全局聚合
records = [('hot', 1)] * 100 + [('a', 1)] * 10 + [('b', 1)] * 5   # 'hot' 倾斜

def two_phase_agg(records, n_rand=3):
    phase1 = {}
    for k, v in records:
        p = f'{k}#{rng.integers(0, n_rand)}'
        phase1[p] = phase1.get(p, 0) + v
    phase2 = {}
    for pk, v in phase1.items():
        k = pk.split('#')[0]
        phase2[k] = phase2.get(k, 0) + v
    return phase2

res = two_phase_agg(records)
print('两阶段聚合结果:', res)
assert res['hot'] == 100 and res['a'] == 10 and res['b'] == 5
print('✅ 两阶段聚合：先按加前缀的 key 分散计算，再合并，结果一致且不再倾斜')

两阶段聚合结果: {'hot': 100, 'a': 10, 'b': 5}
✅ 两阶段聚合：先按加前缀的 key 分散计算，再合并，结果一致且不再倾斜


## 7️⃣ 自测清单

- [ ] 能把"一条数据从产生到应用"讲 3 分钟
- [ ] 能答 18 个高频速答题
- [ ] 能手写 TopN 分治、两阶段聚合
- [ ] 能画出数仓四层与 Kafka 架构
- [ ] 能说出 Flink watermark/窗口/精确一次